In [1]:
# ================================================
# PROJET SQL — Analyse E-commerce Olist avec SQL
# Auteur : Islem | Juillet 2026
# ================================================

import pandas as pd
import sqlite3

# Connexion à la base de données (crée le fichier si inexistant)
conn = sqlite3.connect(r"C:\Users\ISLEM\Documents\olist.db")
print("Connexion OK !")

Connexion OK !


In [2]:
# --- CHARGEMENT DES DONNÉES DANS SQLITE ---
orders = pd.read_csv(r"C:\Users\ISLEM\Documents\olist_orders_dataset.csv")
payments = pd.read_csv(r"C:\Users\ISLEM\Documents\olist_order_payments_dataset.csv")
items = pd.read_csv(r"C:\Users\ISLEM\Documents\olist_order_items_dataset.csv")
products = pd.read_csv(r"C:\Users\ISLEM\Documents\olist_products_dataset.csv")

orders.to_sql('orders', conn, if_exists='replace', index=False)
payments.to_sql('payments', conn, if_exists='replace', index=False)
items.to_sql('items', conn, if_exists='replace', index=False)
products.to_sql('products', conn, if_exists='replace', index=False)

print("Tables créées !")

Tables créées !


In [3]:
# --- REQUÊTE 1 : NOMBRE DE COMMANDES PAR STATUT ---
query = """
SELECT order_status, COUNT(*) as nombre
FROM orders
GROUP BY order_status
ORDER BY nombre DESC
"""

pd.read_sql(query, conn)

,order_status,nombre
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [4]:
# --- REQUÊTE 2 : CA PAR STATUT ---
query = """
SELECT o.order_status, 
       ROUND(SUM(p.payment_value), 2) as chiffre_affaires
FROM orders o
JOIN payments p ON o.order_id = p.order_id
GROUP BY o.order_status
ORDER BY chiffre_affaires DESC
"""

pd.read_sql(query, conn)

,order_status,chiffre_affaires
0,delivered,15422461.77
1,shipped,177213.96
2,canceled,143255.60
3,unavailable,126479.51
4,processing,69394.11
5,invoiced,69137.99
6,created,688.10
7,approved,241.08


In [5]:
# --- REQUÊTE 3 : TOP 10 CATÉGORIES PAR REVENUS ---
query = """
SELECT p.product_category_name,
       ROUND(SUM(i.price), 2) as revenus
FROM items i
JOIN products p ON i.product_id = p.product_id
GROUP BY p.product_category_name
ORDER BY revenus DESC
LIMIT 10
"""

pd.read_sql(query, conn)

,product_category_name,revenus
0,beleza_saude,1258681.34
1,relogios_presentes,1205005.68
2,cama_mesa_banho,1036988.68
3,esporte_lazer,988048.97
4,informatica_acessorios,911954.32
5,moveis_decoracao,729762.49
6,cool_stuff,635290.85
7,utilidades_domesticas,632248.66
8,automotivo,592720.11
9,ferramentas_jardim,485256.46


In [6]:
# --- REQUÊTE 4 : COMMANDES EN RETARD ---
query = """
SELECT COUNT(*) as commandes_en_retard
FROM orders
WHERE order_delivered_customer_date > order_estimated_delivery_date
AND order_delivered_customer_date IS NOT NULL
"""

pd.read_sql(query, conn)

,commandes_en_retard
0,7827


In [7]:
# --- REQUÊTE 5 : TAUX DE LIVRAISON À TEMPS PAR MOIS ---
query = """
SELECT 
    STRFTIME('%Y-%m', order_purchase_timestamp) as mois,
    COUNT(*) as total,
    SUM(CASE WHEN order_delivered_customer_date <= order_estimated_delivery_date 
             THEN 1 ELSE 0 END) as a_temps,
    ROUND(SUM(CASE WHEN order_delivered_customer_date <= order_estimated_delivery_date 
             THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) as pct_a_temps
FROM orders
WHERE order_delivered_customer_date IS NOT NULL
GROUP BY mois
ORDER BY mois
"""

pd.read_sql(query, conn)

,mois,total,a_temps,pct_a_temps
0,2016-09,1,0,0.0
1,2016-10,270,267,98.9
2,2016-12,1,1,100.0
3,2017-01,750,727,96.9
4,2017-02,1653,1600,96.8
5,2017-03,2546,2404,94.4
6,2017-04,2303,2122,92.1
7,2017-05,3545,3417,96.4
8,2017-06,3135,3014,96.1
9,2017-07,3872,3739,96.6


In [8]:
# --- REQUÊTE 6 : HAVING — catégories avec plus de 500k BRL de revenus ---
query = """
SELECT p.product_category_name,
       ROUND(SUM(i.price), 2) as revenus
FROM items i
JOIN products p ON i.product_id = p.product_id
GROUP BY p.product_category_name
HAVING revenus > 500000
ORDER BY revenus DESC
"""

pd.read_sql(query, conn)

,product_category_name,revenus
0,beleza_saude,1258681.34
1,relogios_presentes,1205005.68
2,cama_mesa_banho,1036988.68
3,esporte_lazer,988048.97
4,informatica_acessorios,911954.32
5,moveis_decoracao,729762.49
6,cool_stuff,635290.85
7,utilidades_domesticas,632248.66
8,automotivo,592720.11


In [9]:
# --- REQUÊTE 7 : SUBQUERY — commandes au dessus du panier moyen ---
query = """
SELECT order_id, payment_value
FROM payments
WHERE payment_value > (SELECT AVG(payment_value) FROM payments)
ORDER BY payment_value DESC
LIMIT 10
"""

pd.read_sql(query, conn)

,order_id,payment_value
0,03caa2c082116e1d31e67e9ae3700499,13664.08
1,736e1922ae60d0d6a89247b851902527,7274.88
2,0812eb902a67711a1cb742b3cdaa65ae,6929.31
3,fefacc66af859508bf1a7934eab1e97f,6922.21
4,f5136e38d1a14a4dbd87dff67da82701,6726.66
5,2cc9089445046817a7539d90805e6e5a,6081.54
6,a96610ab360d42a2e5335a3998b4718a,4950.34
7,b4c4b76c642808cbe472a32b86cddc95,4809.44
8,199af31afc78c699f0dbf71fb178d4d4,4764.34
9,8dbc85d1447242f3b127dda390d56e19,4681.78


In [10]:
# --- REQUÊTE 8 : LEFT JOIN — produits sans catégorie ---
query = """
SELECT i.product_id, p.product_category_name
FROM items i
LEFT JOIN products p ON i.product_id = p.product_id
WHERE p.product_category_name IS NULL
LIMIT 10
"""

pd.read_sql(query, conn)

,product_id,product_category_name
0,ff6caf9340512b8bf6d2a2a6df032cfa,None
1,a9c404971d1a5b1cbc2e4070e02731fd,None
2,5a848e4ab52fd5445cdc07aab1c40e48,None
3,41eee23c25f7a574dfaf8d5c151dbb12,None
4,e10758160da97891c2fdcbc35f0f031d,None
5,76d1a1a9d21ab677a61c3ae34b1b352f,None
6,fbb1cfc2810efabf3235eccf4530f4ae,None
7,5a848e4ab52fd5445cdc07aab1c40e48,None
8,9f69acd4da62618a3f6365b732d00ccd,None
9,ea11e700a343582ad56e4c70e966cb36,None
